In [2]:
import json
import pandas as pd
import numpy as np
import os
import torch

from tabpfn_time_series import TabPFNTSPipeline, TabPFNMode
import time

start_time = time.time()

def root_mean_squared_error(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
# ============================================================
# CONFIG
# ============================================================
DAYS_JSON = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\dataset_days.json"
DATA_DIR  = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\DataCleaning\clean"
OUT_DIR   = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs"

countries = ["Germany", "Ireland", "Portugal"]
days = ["day1", "day2", "day3", "day4", "day5"]

# present in CSV but NOT used in this univariate setup
features = [
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "precipitation",
    "direct_radiation",
]


countries = ["Denmark"]

days = ["day1", "day2", "day3", "day4", "day5"]

features = [
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "precipitation",
    "direct_radiation",
    "price_eur_kwh"
]

PRED_LEN = 96
MAX_CONTEXT = 10000

# ============================================================
# LOAD DAY CUTOFFS
# ============================================================
with open(DAYS_JSON, "r") as f:
    dataset_days = json.load(f)

# ============================================================
# INIT TabPFN-TS PIPELINE (LOCAL)
# ============================================================
pipeline = TabPFNTSPipeline(tabpfn_mode=TabPFNMode.LOCAL)

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Using GPU:", torch.cuda.get_device_name(0))

rmse_results = []

for country in countries:
    print("Processing country:", country)

    data_path = rf"{DATA_DIR}\dataset_{country.capitalize()}.csv"
    df = pd.read_csv(data_path, index_col="timestamp", parse_dates=True).sort_index()

    households = [c for c in df.columns if c not in features]

    for day in days:
        print("   Day:", day)

        cutoff = pd.to_datetime(dataset_days[country][day])

        predictions_df_all_households = None
        rmse_households = []

        for household in households:
            # ----------------------------
            # 1) Build context_df (univariate only)
            # ----------------------------
            hist_idx = df.index[df.index < cutoff]
            s_train = df.loc[hist_idx, household].astype(float)

            # skip empty / tiny history
            if s_train.dropna().shape[0] < 10:
                continue

            context_df = pd.DataFrame({
                "item_id": household,
                "timestamp": s_train.index,
                "target": s_train.values,
            }).tail(MAX_CONTEXT).reset_index(drop=True)

            # ----------------------------
            # 2) Predict (no future_df => use prediction_length)
            # ----------------------------
            pred_df = pipeline.predict_df(
                context_df=context_df,
                prediction_length=PRED_LEN,
            )

            pred_df_reset = pred_df.reset_index()  # item_id, timestamp back as columns

            # Forecast timestamps for this household
            ts_pred = pd.to_datetime(pred_df_reset["timestamp"])

            # init global predictions df once per day using the forecast timestamps
            if predictions_df_all_households is None:
                predictions_df_all_households = pd.DataFrame(index=ts_pred)

            # Median forecast column is float 0.5
            y_pred = pred_df_reset[0.5].to_numpy()
            predictions_df_all_households[household] = y_pred

            # ----------------------------
            # 3) RMSE aligned by forecast timestamps
            # ----------------------------
            y_true = df.loc[ts_pred, household].to_numpy()

            # if NaNs in truth window, skip rmse for this household
            if np.isnan(y_true).any():
                continue

            rmse_households.append(root_mean_squared_error(y_true, y_pred))

        if predictions_df_all_households is None or len(rmse_households) == 0:
            print(f"      No predictions produced for {country} {day}. Skipping.")
            continue

        avg_rmse_households = float(np.mean(rmse_households))

        rmse_results.append({
            "country": country,
            "day": day,
            "rmse": avg_rmse_households
        })

        output = rf"{OUT_DIR}\TabPFNTS_UNIV_pred_{day}_{country.capitalize()}.csv"
        os.makedirs(os.path.dirname(output), exist_ok=True)
        predictions_df_all_households.to_csv(output, index=True)
        print("      Saved:", output)

# ============================================================
# SUMMARY
# ============================================================
rmse_df = pd.DataFrame(rmse_results)
print("\nPer-day RMSE:")
print(rmse_df)

print("\nCross-validated RMSE per country (mean over days):")
print(rmse_df.groupby("country")["rmse"].mean())

end_time = time.time()
total_seconds = end_time - start_time
print(f"Total runtime: {total_seconds:.2f} seconds")

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


tabpfn-v2-regressor-2noar4o2.ckpt:   0%|          | 0.00/44.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/37.0 [00:00<?, ?B/s]

Torch: 2.5.1+cu121
CUDA available: True
Using GPU: NVIDIA RTX 4000 Ada Generation
Processing country: Denmark
   Day: day1


GPU 0:: 100%|██████████| 1/1 [00:03<00:00,  3.50s/it]


...

      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\TabPFNTS_UNIV_pred_day3_Denmark.csv
   Day: day4


GPU 0:: 100%|██████████| 1/1 [00:03<00:00,  3.47s/it]


      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\TabPFNTS_UNIV_pred_day4_Denmark.csv
   Day: day5


GPU 0:: 100%|██████████| 1/1 [00:03<00:00,  3.60s/it]

      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\TabPFNTS_UNIV_pred_day5_Denmark.csv

Per-day RMSE:
   country   day      rmse
0  Denmark  day1  1.740387
1  Denmark  day2  0.530752
2  Denmark  day3  0.651159
3  Denmark  day4  0.528282
4  Denmark  day5  1.901408

Cross-validated RMSE per country (mean over days):
country
Denmark    1.070398
Name: rmse, dtype: float64
Total runtime: 165.28 seconds


In [3]:
print(f"Time taken: {total_seconds:.4f} seconds")


file_path = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\time_spend.json"
# 1. Load existing JSON
with open(file_path, "r") as f:
    data = json.load(f)

# 2. Add model inside "Local"
data["Foundational"]["TabPFN"] = total_seconds

# 3. Save back (without disturbing structure)
with open(file_path, "w") as f:
    json.dump(data, f, indent=4)

Time taken: 165.2751 seconds
